# Generate Outlier Reports

This notebook produces motion outlier data and nuisance regressor files from fMRIPrep confounds output. These files are used directly in L1 modeling as nuisance regressors.

It runs two complementary outlier detection approaches:

1. **Auto-motion** (via `auto-motion-fmriprep` R script): data-driven detection based on multiple motion indicators. Configured via `config.R`.
2. **Rule-based**: flags volumes where global signal deviates > 3 SD *or* framewise displacement > 0.75 mm.

**Outputs (all in `derivatives_nocorrection/outlier/`):**
- `outlier_auto.csv` — auto-motion outlier volumes
- `outlier_manual.csv` — rule-based outlier volumes
- `outlier_summary.csv` — count of outlier volumes per subject/task/run
- `regressors/sub-{ID}/` — per-run motion regressor `.tsv` files for L1 modeling
- `outlier_imgs/sub-{ID}_task-{task}_run-{run}.gif` — animated GIFs of outlier volume images

For FD-only QC plots and HTML reports, see `outlier_detection.ipynb`.

#### History
- 27/9/2021 hychan — created
- Refactored for CNLab pipeline documentation

## 1. Imports

In [ ]:
import os
import glob
import warnings

import imageio
import numpy as np
import pandas as pd
from nilearn import plotting, image
from sklearn.preprocessing import scale

warnings.filterwarnings('ignore')

## 2. Configuration

All paths are derived from this notebook's location (`scripts/FMRIPREP/`). The session label and NIfTI path pattern are the only project-specific values that may need updating.

In [ ]:
# ── Session label ─────────────────────────────────────────────────────────────
# The session used in the confounds glob pattern and NIfTI path
SESSION = 't3'    # e.g. 't2', 't3', 'baseline'

# ── Derived paths (do not edit) ───────────────────────────────────────────────
project_dir      = os.path.abspath('../../')
bids_dir         = os.path.join(project_dir, 'data/bids_data')
fmriprep_dir     = os.path.join(bids_dir, 'derivatives_correction')
output_dir       = os.path.join(fmriprep_dir, 'outlier')
auto_motion_path = '/data00/tools/auto-motion-fmriprep'

# NIfTI path pattern for generating outlier images — uses BIDS keys filled at runtime
nii_pattern = os.path.join(
    bids_dir,
    'sub-{sub}/ses-' + SESSION + '/func/sub-{sub}_ses-' + SESSION + '_task-{task}_run-{run}_bold.nii.gz'
)

# ── Outlier detection rules ───────────────────────────────────────────────────
rules = {
    'gs>3':    lambda df: np.abs(scale(df['global_signal'])) > 3,
    'fd>0.75': lambda df: df['framewise_displacement'] > 0.75,
}

for d in [output_dir,
          os.path.join(output_dir, 'regressors'),
          os.path.join(output_dir, 'outlier_imgs')]:
    os.makedirs(d, exist_ok=True)

print(f'fMRIPrep derivatives : {fmriprep_dir}')
print(f'Outlier output       : {output_dir}')
print(f'Session              : {SESSION}')

## 3. Helper Function

In [ ]:
def parse_bids_filename(filepath):
    """Extract BIDS key-value pairs from a confounds filename."""
    parts = os.path.basename(filepath).replace('.tsv', '').split('_')
    out = {}
    for kv in parts:
        if '-' in kv:
            k, v = kv.split('-', 1)
            if k != 'desc':
                out[k] = v
    return out

---
## Part A: Auto-Motion Detection

Runs the `auto-motion-fmriprep` R script using `config.R` for settings. Update `config.R` if your project paths have changed.

> **Requires:** R, the `auto-motion-fmriprep` package at `/data00/tools/auto-motion-fmriprep/`, and a correctly configured `config.R` in this directory.

In [ ]:
config_path = os.path.abspath('config.R')
print(f'Using config: {config_path}')

cmd = f'cd {auto_motion_path} && Rscript auto_motion_fmriprep.R {config_path}'
!{cmd}

In [ ]:
# Collect auto-motion outlier volumes from the R script output
auto_outliers = []

for reg_txt in glob.glob(os.path.join(output_dir, 'auto-motion-fmriprep/sub-*/sub-*_regressors.txt')):
    reg_df    = pd.read_csv(reg_txt, sep='\t')
    bids_vars = parse_bids_filename(reg_txt)
    idx       = np.where(reg_df['trash'] == 1)[0]

    if len(idx) > 0:
        bids_vars['outlier_type'] = 'auto'
        bids_vars['outlier_vol']  = idx
        auto_outliers.append(pd.DataFrame(bids_vars))

if auto_outliers:
    auto_df = pd.concat(auto_outliers, ignore_index=True)
    auto_df.to_csv(os.path.join(output_dir, 'outlier_auto.csv'), index=False)
    print(f'Saved outlier_auto.csv  ({len(auto_df)} flagged volumes)')
else:
    print('No auto-motion outliers found.')

---
## Part B: Rule-Based Outlier Detection

Applies the rules defined in the configuration cell to each run's confounds file. Also generates the **motion regressor `.tsv` files** used in L1 modeling.

In [ ]:
manual_outliers = []
confound_glob   = os.path.join(fmriprep_dir, f'sub-*/ses-{SESSION}/func/*confounds_timeseries.tsv')

for f in sorted(glob.glob(confound_glob)):
    bids_vars = parse_bids_filename(f)
    confounds = pd.read_csv(f, sep='\t')

    # ── Build motion regressor file ───────────────────────────────────────────
    confounds['trash'] = np.where(
        (np.abs(scale(confounds['global_signal'])) > 3) |
        (confounds['framewise_displacement'] > 0.75),
        1, 0
    )
    motion_df = confounds[['trans_x', 'trans_y', 'trans_z',
                            'rot_x',   'rot_y',   'rot_z',
                            'csf', 'trash']].copy()

    # Euclidean distance summary columns
    motion_df['euclidean_trans']       = np.linalg.norm(confounds[['trans_x','trans_y','trans_z']], axis=1)
    motion_df['euclidean_rot']         = np.linalg.norm(confounds[['rot_x','rot_y','rot_z']] * 50, axis=1)
    motion_df['euclidean_trans_deriv'] = motion_df['euclidean_trans'].diff().fillna(0)
    motion_df['euclidean_rot_deriv']   = motion_df['euclidean_rot'].diff().fillna(0)

    # Save motion regressor file next to confounds
    motion_dir  = os.path.join(output_dir, 'regressors', f'sub-{bids_vars["sub"]}')
    os.makedirs(motion_dir, exist_ok=True)
    motion_file = os.path.basename(f).replace('confounds', 'motion')
    motion_df.to_csv(os.path.join(motion_dir, motion_file), sep='\t', index=False)

    # ── Collect rule-based outliers ────────────────────────────────────────────
    for rule_name, rule_fn in rules.items():
        idx = np.where(rule_fn(confounds))[0]
        row = dict(bids_vars)
        row['outlier_type'] = rule_name
        row['outlier_vol']  = idx
        manual_outliers.append(pd.DataFrame(row))

    print('.', end='', flush=True)

manual_df = pd.concat(manual_outliers, ignore_index=True)
manual_df.to_csv(os.path.join(output_dir, 'outlier_manual.csv'), index=False)
print(f'\nSaved outlier_manual.csv  ({len(manual_df)} flagged volumes)')
print(f'Motion regressor files saved to: {os.path.join(output_dir, "regressors")}')

---
## Part C: Generate Outlier GIFs

Reads both outlier CSV files, merges them, and creates an animated GIF per run showing each flagged volume as an orthogonal brain slice. Useful for visually confirming whether outliers correspond to obvious motion or signal artifacts.

In [ ]:
# Load and merge auto + manual outliers
auto_path   = os.path.join(output_dir, 'outlier_auto.csv')
manual_path = os.path.join(output_dir, 'outlier_manual.csv')

parts = []
if os.path.exists(auto_path):   parts.append(pd.read_csv(auto_path))
if os.path.exists(manual_path): parts.append(pd.read_csv(manual_path))

if not parts:
    raise FileNotFoundError('No outlier CSV files found. Run Parts A and B first.')

outlier = (
    pd.concat(parts, ignore_index=True)
    .pivot_table(
        index=['sub', 'task', 'run', 'outlier_vol'],
        values='outlier_type',
        aggfunc=lambda x: ', '.join(x)
    )
    .reset_index()
)

print(f'Total unique outlier volumes: {len(outlier)}')

In [ ]:
gif_dir = os.path.join(output_dir, 'outlier_imgs')
os.makedirs(gif_dir, exist_ok=True)

for (sub, task, run), run_df in outlier.groupby(['sub', 'task', 'run']):
    nii_path = nii_pattern.format(sub=sub, task=task, run=int(run))

    if not os.path.exists(nii_path):
        print(f'NIfTI not found, skipping: {nii_path}')
        continue

    print(f'Generating GIF: sub-{sub} task-{task} run-{run} ({len(run_df)} outlier vols)')

    outlier_imgs = image.index_img(nii_path, run_df['outlier_vol'])
    frames = []

    for img, vol, otype in zip(image.iter_img(outlier_imgs),
                                run_df['outlier_vol'],
                                run_df['outlier_type']):
        tmp = f'_tmp_vol{vol:04d}.png'
        plotting.plot_anat(
            anat_img=img, cut_coords=[0, 0, 0],
            output_file=tmp, display_mode='ortho',
            title=f'Vol {vol:04d} — {otype}',
            annotate=False, draw_cross=False,
            black_bg='auto', dim='auto'
        )
        frames.append(imageio.imread(tmp))
        os.remove(tmp)

    gif_path = os.path.join(gif_dir, f'sub-{sub}_task-{task}_run-{run}.gif')
    imageio.mimsave(gif_path, frames, fps=1)
    print(f'  Saved: {gif_path}')

print('\nGIF generation complete.')

---
## Part D: Outlier Summary

In [ ]:
summary = (
    outlier
    .pivot_table(index=['sub', 'task', 'run'], values='outlier_vol', aggfunc='count')
    .reset_index()
    .rename(columns={'outlier_vol': 'n_outlier_vols'})
    .pivot(index='sub', columns=['task', 'run'], values='n_outlier_vols')
    .fillna(0)
    .astype(int)
)

summary_path = os.path.join(output_dir, 'outlier_summary.csv')
summary.to_csv(summary_path)
print(f'Saved: {summary_path}')
summary